# Winding the Fourier transform

*An interactive companion to Chapter 5, "What is the Fourier transform doing?"*

The Fourier transform probes a signal $x(t)$ with a phasor $e^{-j 2\pi g t}$ at a
**probe frequency** $g$, multiplies the two, and adds up the result over time. Multiplying
by the phasor **winds** the signal around the complex plane, and the **center of mass** of
the wound-up curve measures how much of frequency $g$ is present in $x(t)$.

The signal here is

$$x(t) = \sin(2\pi\,2\,t) + \tfrac{1}{2}\sin(2\pi\,4\,t),$$

a <span style="color:#0271AE">2 Hz tone</span> plus a quieter
<span style="color:#9B5DE5">4 Hz tone</span> at half the amplitude. The animation **sweeps
the probe frequency** $g$:

- **left** &mdash; the wound product, split into its
  <span style="color:#0271AE">2 Hz part</span> and
  <span style="color:#9B5DE5">4 Hz part</span>. Each part's center of mass is drawn as an
  arrow, and the <span style="color:#DC2830">red dot</span> is their sum: the center of mass
  of the whole signal.
- **top right** &mdash; the signal and its two tones (the same colors), with the probe
  $\sin(2\pi g t)$ drawn as an <span style="color:#F2AA00">orange dashed wave</span> whose
  wavelength shrinks as the probe frequency $g$ rises. It shares the signal's phase, so it
  locks onto a tone exactly when $g$ matches it.
- **bottom right** &mdash; the magnitude of the center of mass at each probe frequency: the
  amplitude spectrum. It spikes <span style="color:#0271AE">blue at 2 Hz</span> and, half as
  tall, <span style="color:#9B5DE5">purple at 4 Hz</span> &mdash; exactly the amplitudes in
  the equation.

Press **play**. Each tone's contribution swings out only when the probe matches it; otherwise
it is balanced around the origin and cancels.


In [ ]:
# hide
# --- Dependencies -----------------------------------------------------------
# All imports and installs for this page live here. The installs are no-ops
# when the packages are already present (they ship with the Jupyter Book build
# environment), so this cell just needs to run once.
%pip install -q numpy matplotlib myst-nb

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.animation import FuncAnimation
from myst_nb import glue
from IPython.display import HTML


In [ ]:
# hide
plt.rcParams.update({
    "figure.dpi": 64,                # keeps the embedded jshtml a sane size
    "animation.html": "jshtml",
    "text.usetex": False,            # mathtext: no LaTeX install needed locally
    "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
})

C2, C4, CT, CP = "#0271AE", "#9B5DE5", "#DC2830", "#F2AA00"   # 2Hz, 4Hz, total, probe

# --- A two-tone signal: actual sines, accurate amplitudes (more 2 Hz) ------
T, n_pts = 2.0, 400
t = np.linspace(0.0, T, n_pts, endpoint=False)
A2, F2, A4, F4 = 1.0, 2.0, 0.5, 4.0
x2 = A2 * np.sin(2 * np.pi * F2 * t)
x4 = A4 * np.sin(2 * np.pi * F4 * t)
x = x2 + x4

# --- Sweep the probe frequency; wound components and centers of mass -------
frame_rate, n_frames = 24, 120
g = np.linspace(0.5, 6.0, n_frames)
ph = np.exp(-1j * 2 * np.pi * np.outer(g, t))
w2, w4 = x2[None, :] * ph, x4[None, :] * ph
com2, com4 = w2.mean(1), w4.mean(1)
com = com2 + com4
mag, mag2, mag4 = np.abs(com), np.abs(com2), np.abs(com4)
ymax = mag.max() * 1.3

# Tone frequencies and their spectrum heights, for the "discovery" stars.
peaks = []
for f, c in [(F2, C2), (F4, C4)]:
    k = int(np.argmin(np.abs(g - f)))
    lo, hi = max(0, k - 5), min(n_frames, k + 6)
    kk = lo + int(np.argmax(mag[lo:hi]))
    peaks.append((g[kk], mag[kk], c, f"{f:g} Hz"))

# --- Layout: winding (big, left) | signal (top-right) / spectrum (bottom) --
fig = plt.figure(figsize=(10.5, 5.6))
gs = gridspec.GridSpec(2, 2, width_ratios=[1.25, 1], height_ratios=[1, 1.55],
                       hspace=0.55, wspace=0.3)
ax_w = fig.add_subplot(gs[:, 0])
ax_sig = fig.add_subplot(gs[0, 1])
ax_s = fig.add_subplot(gs[1, 1])

# signal panel: color-coded tones (the equation) plus the animated dashed PROBE,
# whose wavelength shrinks as the probe frequency rises (the 3Blue1Brown effect).
ax_sig.plot(t, x, color="#343837", lw=2.0, label=r"$x(t)$", zorder=3)
ax_sig.plot(t, x2, color=C2, lw=1.3, alpha=0.7, label=r"$\sin(2\pi\,2\,t)$", zorder=1)
ax_sig.plot(t, x4, color=C4, lw=1.3, alpha=0.7, label=r"$\frac{1}{2}\sin(2\pi\,4\,t)$", zorder=1)
probe_line, = ax_sig.plot(t, np.sin(2 * np.pi * g[0] * t), color=CP, lw=1.8,
                          ls="--", alpha=0.95, label=r"probe $\sin(2\pi g\,t)$", zorder=4)
ax_sig.set_xlim(0, T); ax_sig.set_ylim(-1.8, 2.3); ax_sig.set_yticks([])
ax_sig.set_xlabel("time (s)")
ax_sig.legend(loc="upper center", fontsize=8.5, labelcolor="linecolor", ncol=2,
              handlelength=1.3, columnspacing=1.0, borderpad=0.3,
              bbox_to_anchor=(0.5, 1.34)).get_frame().set_alpha(0.0)

theta = np.linspace(0, 2 * np.pi, 220)


def _arrow(p0, p1, color):
    ax_w.annotate("", xy=(p1.real, p1.imag), xytext=(p0.real, p0.imag),
                  arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2,
                                  shrinkA=0, shrinkB=0))


def draw_winding(n):
    ax_w.clear()
    ax_w.set_xlim(-1.18, 1.18); ax_w.set_ylim(-1.18, 1.18)
    ax_w.set_aspect("equal", "box")
    ax_w.plot(np.cos(theta), np.sin(theta), color="0.9", lw=1, zorder=0)
    ax_w.axhline(0, color="0.85", lw=1, zorder=0)
    ax_w.axvline(0, color="0.85", lw=1, zorder=0)
    ax_w.plot(w2[n].real, w2[n].imag, color=C2, lw=1.6, alpha=0.55, zorder=2)
    ax_w.plot(w4[n].real, w4[n].imag, color=C4, lw=1.6, alpha=0.55, zorder=2)
    c2, ct = com2[n], com[n]
    _arrow(0 + 0j, c2, C2)                  # 2 Hz contribution
    _arrow(c2, ct, C4)                       # 4 Hz contribution, tip-to-tail
    ax_w.plot(ct.real, ct.imag, "o", color=CT, ms=22, alpha=0.22, zorder=4)
    ax_w.plot(ct.real, ct.imag, "o", color=CT, ms=10, mec="white", mew=1.3, zorder=5)
    ax_w.set_xlabel("real"); ax_w.set_ylabel("imaginary")
    ax_w.set_title(rf"probe  $g$ = {g[n]:.2f} Hz", fontsize=12.5)


def draw_spectrum(n):
    ax_s.clear()
    gg = g[:n + 1]
    ax_s.fill_between(gg, mag[:n + 1], color="#343837", alpha=0.10)
    ax_s.plot(gg, mag2[:n + 1], color=C2, lw=1.4, alpha=0.8)
    ax_s.plot(gg, mag4[:n + 1], color=C4, lw=1.4, alpha=0.8)
    ax_s.plot(gg, mag[:n + 1], color="#343837", lw=2)
    ax_s.plot(g[n], mag[n], "o", color=CT, ms=8, mec="white", zorder=6)
    for gp, yp, c, label in peaks:
        if g[n] >= gp - 0.01:
            ax_s.plot(gp, yp, "*", color=c, ms=16, zorder=7)
            ax_s.annotate(label, (gp, yp), textcoords="offset points", xytext=(0, 9),
                          ha="center", color=c, fontsize=11, fontweight="bold")
    ax_s.set_xlim(g[0], g[-1]); ax_s.set_ylim(0, ymax)
    ax_s.set_xlabel(r"probe frequency  $g$  (Hz)")
    ax_s.set_ylabel("|center of mass|")


def animate(n):
    probe_line.set_ydata(np.sin(2 * np.pi * g[n] * t))   # probe period shrinks with g
    draw_winding(n)
    draw_spectrum(n)
    return ()


anim = FuncAnimation(fig, animate, frames=np.arange(n_frames),
                     interval=1000 / frame_rate, blit=False)
player = HTML(anim.to_jshtml())
plt.close(fig)

# Register for the book page (embed with `{glue}`winding-sweep``); the player
# below previews the animation inline here in the notebook.
glue("winding-sweep", player, display=False)
player


**What to watch for.** Each tone contributes its own winding. The
<span style="color:#0271AE">2 Hz part</span> only lines up &mdash; pulling its center of mass
far from the origin &mdash; when the probe passes 2 Hz; the <span style="color:#9B5DE5">4 Hz
part</span> only lines up at 4 Hz. The red dot, the center of mass of the whole signal, is just
the sum of the two colored arrows. Sweeping the probe across every frequency and recording how
far that red dot swings out is exactly what the Fourier transform does, and the result is the
amplitude spectrum.
